In [2]:
import numpy as np

#### 1. Example UNIFAC Smith Van Ness Appendix G

* Binary system diethylamine (1) /n-heptane (2)
* This calculation is done in molar fractions, for larger molecules like polymers of triacylglycerides, mass compositions must be used.

In [3]:
molar_x_1 = 0.4
molar_x_2 = 0.6

The subgroup data is organized in the following order:

1. k: subgroup identification number (This is an assigned number for the table)
2. Rk: Group area volume
3. Qk: Group area parameter
4. v^1_k: Number of times that group k is in the molecule 1
5. v^2_k: Number of times that group k is in the molecule 2

In [4]:
subgroup_data = {
    
   'CH3': [1, 0.9011, 0.848, 2, 2], 
   'CH2': [1, 0.6744, 0.540, 1, 5],  
   'CH2NH': [33, 1.2070, 0.936, 1, 0]
}

In [5]:
# r1,2,n.. = ri, is the molecular volume
r1 = subgroup_data['CH3'][3] * subgroup_data['CH3'][1] + subgroup_data['CH2'][3] * subgroup_data['CH2'][1] + subgroup_data['CH2NH'][3] * subgroup_data['CH2NH'][1]
r2 = subgroup_data['CH3'][4] * subgroup_data['CH3'][1] + subgroup_data['CH2'][4] * subgroup_data['CH2'][1] + subgroup_data['CH2NH'][4] * subgroup_data['CH2NH'][1]

In [6]:
# q1,2,n.. = qi, is the molecular area
q1 = subgroup_data['CH3'][3] * subgroup_data['CH3'][2] + subgroup_data['CH2'][3] * subgroup_data['CH2'][2] + subgroup_data['CH2NH'][3] * subgroup_data['CH2NH'][2]
q2 = subgroup_data['CH3'][4] * subgroup_data['CH3'][2] + subgroup_data['CH2'][4] * subgroup_data['CH2'][2] + subgroup_data['CH2NH'][4] * subgroup_data['CH2NH'][2]

In [17]:
# eki, k is the subgroup, i is the molecule. eki is the fraction of superficial area that each subgroup adds to each molecule

e11 = subgroup_data['CH3'][3]*subgroup_data['CH3'][2]/q1
e12 = subgroup_data['CH3'][4]*subgroup_data['CH3'][2]/q2

e21 = subgroup_data['CH2'][3]*subgroup_data['CH2'][2]/q1
e22 = subgroup_data['CH2'][4]*subgroup_data['CH2'][2]/q2

e331 = subgroup_data['CH2NH'][3]*subgroup_data['CH2NH'][2]/q1
e332 = subgroup_data['CH2NH'][4]*subgroup_data['CH2NH'][2]/q1

eki = np.array([e11, e12, e21, e22, e331, e332])

In [9]:
# GIP from the literature, in k
a11 = 0
a12 = 0
a21 = 0
a22 = 0
a3333 = 0
a133 = 255.7
a233 = 255.7
a331 = 65.33
a332 = 65.33

GIP = np.array([a11, a12, a21, a22, a3333, a133, a233, a331, a332])

In [10]:
# Interaction energies
tao = np.exp(-GIP/308.15)
tao

array([1.        , 1.        , 1.        , 1.        , 1.        ,
       0.43614057, 0.43614057, 0.80895892, 0.80895892])

Bik = Summation(emi*taomk), m is a subindex that goes over all the subgroups, so if we take the subgroup k = 1 and component i = 1 we would get:

B1,1 = e1,1*tao1,1 + e2,1*tao2,1 + e33,1*tao33,1
B1,1 = 0.5347*1 + 0.1702*1 + 0.2951*0.8091

B1,1 = 0.9436

In [28]:
# Here we calculate the Bik
B11 = e11*tao[0] + e21*tao[2] + e331*tao[7]
B12 = e11*tao[1] + e21*tao[3] + e331*tao[8]
B133 = e11*tao[5] + e21*tao[6] + e331*tao[4]

B21 = e12*tao[0] + e22*tao[2] + e332*tao[7]
B22 = e12*tao[1] + e22*tao[3] + e332*tao[8]
B233 = e12*tao[5] + e22*tao[6] + e332*tao[4]

In [48]:
# Here we calcualte the thetak, j is a subindex that goes over all the molecules/components. This is an average effect of each subgroup in the overall area of the mixture.

q_denominator = molar_x_1*q1 + molar_x_2*q2
r_denominator = molar_x_1*r1 + molar_x_2*r2

theta1 = (molar_x_1*q1*e11 + molar_x_2*q2*e12 )/q_denominator

theta2 = (molar_x_1*q1*e21 + molar_x_2*q2*e22 )/q_denominator

theta3 = (molar_x_1*q1*e331 + molar_x_2*q2*e332 )/q_denominator


In [45]:
# Here we calculate the sk, weighted effect of each interaction energy with the average superficial area.

s1 = theta1*tao[0] + theta2*tao[2] + theta3*tao[7]

s2 = theta1*tao[1] + theta2*tao[3] + theta3*tao[8]

s33 = theta1*tao[5] + theta2*tao[6] + theta3*tao[4]

Calculate combinatorial part

In [54]:
J1 = r1/r_denominator
L1 = q1/q_denominator

lngamma1c = 1 - J1 + np.log(J1) - 5*q1*(1-(J1/L1) + np.log(J1/L1))


J2 = r2/r_denominator
L2 = q2/q_denominator

lngamma2c = 1 - J2 + np.log(J2) - 5*q1*(1-(J2/L2) + np.log(J2/L2))

Calculate residual part

In [61]:
lngamma1r = q1*(1 - ((theta1*B11/s1 - e11*np.log(B11/s1)) + (theta2*B12/s2 - e21*np.log(B12/s2)) + (theta3*B133/s33 - e331*np.log(B133/s33))))

lngamma2r = q2*(1 - ((theta1*B21/s1 - e12*np.log(B21/s1)) + (theta2*B22/s2 - e22*np.log(B22/s2)) + (theta3*B233/s33 - e332*np.log(B233/s33))))

In [63]:
lngamma1 = lngamma1c + lngamma1r
gamma1 = np.exp(lngamma1)

lngamma2 = lngamma2c + lngamma2r
gamma2 = np.exp(lngamma2)

In [64]:
gamma1

1.1330392999346754

In [65]:
gamma2

1.0469630934098826